# Multimodal Question Answering with a Custom LLaVA Architecture

A LLaVA-style vision-language model for multiple-choice question answering over images, built by connecting a **LongCLIP ViT-B/32** vision encoder to the **Qwen3-0.6B** language model through a trainable projector.

Developed for the ECS7001P Neural Networks & NLP module at Queen Mary University of London (2026). See the repository README for a summary of the approach and results.


In [ ]:
!pip install transformers

**Dataset**

11,600 images, each paired with 5–6 multiple-choice questions (~65k questions in total). The dataset was provided by the module and is not redistributed here.

* **Training set:** 10k images and 56k questions (image–question alignment provided)
* **Validation set:** 600 images and 3.3k questions (alignment must be recovered, see Task 1)

Files: `images/train/`, `images/test_val/`, `vqa_*_student.jsonl` (questions and options) and `caption_*_student.jsonl` (captions used to map images back to questions).


In [ ]:
# Download the module-provided dataset and unzip it into the working directory
# so that ./images/ and the vqa_*.jsonl / caption_*.jsonl files are available.


In [ ]:
import torch, os, json
from transformers import CLIPModel, CLIPProcessor, CLIPVisionConfig
from transformers.image_utils import load_image

**Components**

* **Vision encoder:** AlpachinoNLP/LongCLIP-ViT-B-32
* **Language model:** Qwen/Qwen3-0.6B
* No other pretrained models are used.


In [ ]:
CLIP_MODEL_ID = "AlpachinoNLP/LongCLIP-ViT-B-32"
TEXI_MODEL_ID = "Qwen/Qwen3-0.6B"

## Task 1: Recovering image–question mappings

Validation images are provided in bulk without their question mapping. `recover_image_id()` embeds every image with the LongCLIP vision encoder and every caption with the LongCLIP text encoder, matches each caption to its most similar image, and uses the `caption_ids` field to attach the recovered image path to each question. The result is saved as `vqa_val_student_with_image_ids.jsonl` and checked with `evaluate_image_caption_alignment()`.


In [ ]:
def evaluate_image_caption_alignment(gold_path,pred_path):
  gold_data = set()
  pred_data = set()
  for line in open(gold_path, 'r'):
    jdata = json.loads(line)
    gold_data.add((jdata['caption_id'],jdata['image']))
  for line in open(pred_path, 'r'):
    jdata = json.loads(line)
    pred_data.add((jdata['caption_id'],jdata['image']))
  return len(gold_data & pred_data)/len(gold_data)

In [ ]:
import glob
from PIL import Image


def _encode(projection, pooler_output):
    """Applies the CLIP projection head and L2-normalizes so that a dot
    product between two encoded vectors equals cosine similarity."""
    feats = projection(pooler_output)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats


def _get_caption_id(entry):
    """The vqa_*_student.jsonl entries link to captions via a 'caption_ids'
    key (per the assignment spec). Some corpus versions store a single id,
    others a list of ids for the same image - handle both, taking the first
    id when a list is given. Falls back to 'caption_id' (singular) if that
    is what the file actually contains, since this varies by export."""
    key = 'caption_ids' if 'caption_ids' in entry else 'caption_id'
    cid = entry[key]
    return cid[0] if isinstance(cid, list) else cid


def recover_image_id():
  """
  Recovers the `image` field in vqa_val_student.jsonl by semantically
  aligning validation images (images/test_val/*.jpg) with their captions
  using LongCLIP embeddings, then propagating the match to every MCQ that
  references that caption via caption_ids.

  Steps:
    1. Load LongCLIP (vision + text towers).
    2. Embed every image in images/test_val/ in batches.
    3. Embed every caption in caption_val_student.jsonl in batches.
    4. For each caption, pick the image with the highest cosine similarity.
    5. Map every MCQ (via its caption_ids) back to the recovered image path.
    6. Write vqa_val_student_with_image_ids.jsonl with the new 'image' field.
  """
  device = 'cuda' if torch.cuda.is_available() else 'cpu'

  # 1) Load LongCLIP
  model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device).eval()
  processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)

  # 2) Collect all validation images and extract embeddings
  image_paths = sorted(glob.glob('images/test_val/*.jpg'))
  image_embeds = []
  with torch.no_grad():
      for i in range(0, len(image_paths), 64):
          batch_paths = image_paths[i:i + 64]
          images = [Image.open(p).convert('RGB') for p in batch_paths]
          inputs = processor(images=images, return_tensors='pt').to(device)
          vision_out = model.vision_model(**inputs)
          feats = _encode(model.visual_projection, vision_out.pooler_output)
          image_embeds.append(feats.cpu())
  image_embeds = torch.cat(image_embeds, dim=0)
  print(f"Image embedding shape: {image_embeds.shape}")

  # 3) Load validation captions and embed them
  cap_path = 'caption_val_student.jsonl'
  vqa_in = 'vqa_val_student.jsonl'
  vqa_out = 'vqa_val_student_with_image_ids.jsonl'

  cap_ids, cap_texts = [], []
  with open(cap_path) as f:
      for line in f:
          d = json.loads(line)
          cap_ids.append(d['caption_id'])
          cap_texts.append(d['caption'])

  text_embeds = []
  with torch.no_grad():
      for i in range(0, len(cap_texts), 64):
          batch = cap_texts[i:i + 64]
          inputs = processor(text=batch, return_tensors='pt', padding=True, truncation=True).to(device)
          text_out = model.text_model(**inputs)
          feats = _encode(model.text_projection, text_out.pooler_output)
          text_embeds.append(feats.cpu())
  text_embeds = torch.cat(text_embeds, dim=0)
  print(f"Caption embedding shape: {text_embeds.shape}")

  # 4) Cosine similarity: each caption -> best matching image
  sims = text_embeds @ image_embeds.T
  best_idx = sims.argmax(dim=1).tolist()
  cap2img = {cap_ids[i]: image_paths[best_idx[i]] for i in range(len(cap_ids))}

  # 5) Recover the image field for every MCQ via its caption_ids
  with open(vqa_in) as fin, open(vqa_out, 'w') as fout:
      for line in fin:
          d = json.loads(line)
          cid = _get_caption_id(d)
          d['image'] = cap2img[cid]
          fout.write(json.dumps(d) + '\n')

  # 6) Evaluation
  acc = evaluate_image_caption_alignment('vqa_val_teacher.jsonl','vqa_val_student_with_image_ids.jsonl')
  print(f"Evaluation accuracy: {acc*100.0:.2f}%")

In [ ]:
recover_image_id()

config.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  606MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/712 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/588 [00:00<?, ?B/s]

Image embedding shape: torch.Size([1600, 512])
Caption embedding shape: torch.Size([600, 512])
Evaluation accuracy: 99.83%


## Task 2: Building and training the LLaVA model

1. **Model construction:** extract the LongCLIP vision tower and combine it with Qwen3.
2. **Alignment:** `LlavaMultiModalProjector` maps CLIP visual tokens into the Qwen3 embedding space.
3. **Training:** the model reads an image and a multiple-choice question and learns to predict the correct option.
4. **Inference:** `get_prediction()` scores options from the logits of the answer letters instead of generating free text; `evaluation()` computes validation accuracy.


In [ ]:
import random,json,datetime
import torch
import numpy as np
from PIL import Image
from torch import nn
from transformers import (AutoTokenizer,
                          CLIPImageProcessor,
                          AutoModelForCausalLM,
                          CLIPVisionModel,
                          LlavaForConditionalGeneration,
                          LlavaProcessor,
                          CLIPVisionConfig,
                          LlavaConfig)
from transformers.activations import ACT2FN

In [ ]:
#evaluate the vqa accuracy between the gold and predicted data
def evaluation(gold_path,pred_path):
  gold_data = []
  pred_data = []
  for line in open(gold_path, 'r'):
    jdata = json.loads(line)
    gold_data.append(jdata['answer'])
  for line in open(pred_path, 'r'):
    jdata = json.loads(line)
    pred_data.append(jdata['answer'])
  return sum(1 for i in range(len(gold_data)) if gold_data[i] == pred_data[i])/len(gold_data)


#get the predictions from the model for evaluation
def get_prediction(model,processor,batch_size,device,dataset):
  """
  Returns predictions from the model. For each sample, outputs an index
  0-3 corresponding to A/B/C/D.

  Strategy: instead of free-form generation, compare the logits of the A,
  B, C, D tokens at the last non-pad position and pick the highest one.
  This is faster and more reliable for MCQ-style evaluation than sampling
  text and parsing it.
  """
  model.eval()
  image_paths = dataset["image_paths"]
  prompts     = dataset["prompts"]
  n = len(prompts)

  tokenizer = processor.tokenizer
  tokenizer.padding_side = "right"  # last_idx below assumes right-padding
  abcd_token_ids = []
  for letter in ["A", "B", "C", "D"]:
      ids = tokenizer(letter, add_special_tokens=False).input_ids
      assert len(ids) == 1, f"{letter!r} is not a single token: {ids}"
      abcd_token_ids.append(ids[0])

  predictions = []
  with torch.no_grad():
      for start in range(0, n, batch_size):
          end = min(start + batch_size, n)
          batch_prompts = prompts[start:end]
          batch_images  = [Image.open(p).convert('RGB') for p in image_paths[start:end]]

          inputs = processor(
              images=batch_images,
              text=batch_prompts,
              return_tensors='pt',
              padding=True,
              truncation=True,
          )
          inputs = {k: v.to(device) for k, v in inputs.items()}

          outputs = model(**inputs)
          logits = outputs.logits  # [batch, seq_len, vocab_size]

          attention_mask = inputs["attention_mask"]
          last_idx = attention_mask.sum(dim=1) - 1

          for i in range(logits.size(0)):
              last_logits = logits[i, last_idx[i], :]
              abcd_logits = torch.tensor([last_logits[tid].item() for tid in abcd_token_ids])
              pred_idx = int(abcd_logits.argmax().item())  # 0, 1, 2, or 3
              predictions.append(pred_idx)

  return predictions


#output the predictions to the files
def write_answers_to_file(path, predictions, answer_to_ids):
  """
  path:          input jsonl file (e.g., 'vqa_val_student_with_image_ids.jsonl')
  predictions:   List[int], 0/1/2/3 (A/B/C/D index) for each sample, in the
                 same order as the lines in `path`.
  answer_to_ids: dict like {'A': 0, 'B': 1, 'C': 2, 'D': 3}

  Output: same as input path, but with '_with_image_ids' replaced by
  '_final' (e.g. vqa_val_student_with_image_ids.jsonl -> vqa_val_student_final.jsonl).
  """
  ids_to_answer = {v: k for k, v in answer_to_ids.items()}
  out_path = path.replace('_with_image_ids.jsonl', '_final.jsonl')

  with open(path, 'r') as fin, open(out_path, 'w') as fout:
      for i, line in enumerate(fin):
          d = json.loads(line)
          pred_idx = predictions[i]
          d['answer'] = ids_to_answer[pred_idx]
          fout.write(json.dumps(d) + '\n')

  print(f"Written: {out_path} ({len(predictions)} samples)")

In [ ]:
#The projection layer for LLaVA
class LlavaMultiModalProjector(nn.Module):
    """
    Maps CLIP visual features into the Qwen3 embedding space.

    A 2-layer MLP with GELU activation, following the LLaVA-1.5 design.
    The first linear layer changes dimension from the vision hidden_size
    (768) to the text hidden_size, and the second layer keeps the
    dimension while adding non-linear capacity for cross-modal alignment.
    """

    def __init__(self, config: LlavaConfig):
        super().__init__()
        self.linear_1 = nn.Linear(
            config.vision_config.hidden_size,
            config.text_config.hidden_size,
            bias=True,
        )
        self.act = ACT2FN[config.projector_hidden_act]
        self.linear_2 = nn.Linear(
            config.text_config.hidden_size,
            config.text_config.hidden_size,
            bias=True,
        )

    def forward(self, image_features):
        x = self.linear_1(image_features)
        x = self.act(x)
        x = self.linear_2(x)
        return x

In [ ]:
#load the image paths and create prompt from MCQ
def load_image_vqa_data(path,answer_to_ids):
  image_paths = []
  prompts = []
  answers = []
  for line in open(path, 'r'):
    jdata = json.loads(line)
    image_paths.append(jdata['image'])
    question = jdata['question']
    options = jdata['options']
    answer = answer_to_ids.get(jdata['answer'],None)

    prompts.append("<image>\nAnswer with A, B, C, or D only.\n"
        f"Question: {question}\n"
        "Options:\n"
        f"A. {options[0]}\n"
        f"B. {options[1]}\n"
        f"C. {options[2]}\n"
        f"D. {options[3]}\n"
        "Answer:\n")
    answers.append(answer)
  return {"image_paths":image_paths, "prompts":prompts, "answers":answers}


class CustomLlava(nn.Module):
    """
    Minimal LLaVA-style wrapper: a frozen CLIP vision tower, a trainable
    projector (any nn.Module mapping vision hidden_size -> text hidden_size,
    same number of tokens in/out), and a causal LM. <image> placeholder
    tokens in the prompt are replaced by the projected visual tokens before
    the language model's forward pass.
    """
    def __init__(self, vision_tower, language_model, projector, image_token_id,
                 vision_feature_layer=-2):
        super().__init__()
        self.vision_tower = vision_tower
        self.language_model = language_model
        self.projector = projector
        self.image_token_id = image_token_id
        self.vision_feature_layer = vision_feature_layer

    def encode_images(self, pixel_values):
        out = self.vision_tower(pixel_values, output_hidden_states=True)
        # second-to-last hidden state, drop the CLS token
        feats = out.hidden_states[self.vision_feature_layer]
        feats = feats[:, 1:, :]
        # find the projector's first Linear layer regardless of architecture,
        # so the dtype cast below works for both the basic and advanced projector
        first_linear = next(m for m in self.projector.modules() if isinstance(m, nn.Linear))
        feats = self.projector(feats.to(first_linear.weight.dtype))
        return feats

    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        # 1) text embeddings
        embed_layer = self.language_model.get_input_embeddings()
        text_embeds = embed_layer(input_ids)

        # 2) image embeddings
        image_embeds = self.encode_images(pixel_values)
        image_embeds = image_embeds.to(text_embeds.dtype)

        # 3) replace <image> tokens with image embeddings
        new_embeds = text_embeds.clone()
        for b in range(input_ids.size(0)):
            mask = (input_ids[b] == self.image_token_id)
            n_img_tokens = mask.sum().item()
            if n_img_tokens == 0:
                continue
            new_embeds[b, mask] = image_embeds[b, :n_img_tokens, :].to(new_embeds.dtype)

        # 4) language model forward
        outputs = self.language_model(
            inputs_embeds=new_embeds,
            attention_mask=attention_mask,
            labels=labels,
        )
        return outputs


class _Processor:
    """Thin callable wrapper so get_prediction() can call processor(images=...,
    text=..., ...) the same way it would call a HF processor, while we
    actually route through our own tokenizer + CLIPImageProcessor pipeline."""
    def __init__(self, tokenizer, call_fn):
        self.tokenizer = tokenizer
        self._call_fn = call_fn

    def __call__(self, images, text, return_tensors='pt', padding=True, truncation=True):
        return self._call_fn(images, text)


def train_evaluate_LLaVA_model():
  random.seed(0)
  torch.manual_seed(0)
  np.random.seed(0)

  device = 'cuda' if torch.cuda.is_available() else 'cpu'
  dtype = torch.float16 if device == 'cuda' else torch.float32
  batch_size = 16  # fits on this run's GPU directly, no gradient accumulation needed
  answer_to_ids = {'A': 0, 'B': 1, 'C': 2, 'D': 3}

  print(f"Device: {device}, dtype: {dtype}")

  # ---- 1) Vision tower (LongCLIP) ----
  print("\n[1/4] Loading vision tower (LongCLIP)...")
  vision_tower = CLIPVisionModel.from_pretrained(CLIP_MODEL_ID, torch_dtype=dtype)
  image_processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL_ID)

  # ---- 2) Language model (Qwen3-0.6B) ----
  print("[2/4] Loading language model (Qwen3-0.6B)...")
  language_model = AutoModelForCausalLM.from_pretrained(TEXI_MODEL_ID, torch_dtype=dtype)
  tokenizer = AutoTokenizer.from_pretrained(TEXI_MODEL_ID)

  if "<image>" not in tokenizer.get_vocab():
      tokenizer.add_special_tokens({"additional_special_tokens": ["<image>"]})
      language_model.resize_token_embeddings(len(tokenizer))
  image_token_id = tokenizer.convert_tokens_to_ids("<image>")

  if tokenizer.pad_token is None:
      tokenizer.pad_token = tokenizer.eos_token
  tokenizer.padding_side = "right"  # get_prediction()'s last_idx assumes this

  img_size = vision_tower.config.image_size
  patch_size = vision_tower.config.patch_size
  num_patches = (img_size // patch_size) ** 2  # 49 for ViT-B/32 at 224px
  print(f"   image_size={img_size}, patch_size={patch_size}, num_patches={num_patches}")

  # ---- 3) Build custom LLaVA model ----
  print("[3/4] Building custom LLaVA model...")

  class _Cfg:
      pass
  proj_cfg = _Cfg()
  proj_cfg.vision_config = vision_tower.config
  proj_cfg.text_config = language_model.config
  proj_cfg.projector_hidden_act = "gelu"

  projector = LlavaMultiModalProjector(proj_cfg)

  # Sanity check
  with torch.no_grad():
      dummy = torch.randn(1, 3, img_size, img_size, dtype=dtype)
      v_out = vision_tower(dummy, output_hidden_states=True)
      feat = v_out.hidden_states[-2]
      print(f"   Vision hidden_states[-2] shape: {feat.shape}  (expected [1, 50, 768])")

  vision_tower = vision_tower.to(device).to(dtype)
  language_model = language_model.to(device).to(dtype)
  projector = projector.to(device).to(torch.float32)

  model = CustomLlava(vision_tower, language_model, projector, image_token_id,
                      vision_feature_layer=-2)

  # PARAMETER FREEZING: only the projector is trainable in the baseline run
  for p in model.vision_tower.parameters():
      p.requires_grad = False
  for p in model.language_model.parameters():
      p.requires_grad = False
  for p in model.projector.parameters():
      p.requires_grad = True

  trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
  total = sum(p.numel() for p in model.parameters())
  print(f"   Trainable parameters: {trainable/1e6:.2f}M / {total/1e6:.2f}M total")

  # ---- Processor wrapper (tokenizer + image processor combined) ----
  image_token_str = "<image>"
  image_tokens_expanded = image_token_str * num_patches  # 49 copies

  def my_processor(images, texts, padding=True, truncation=True):
      texts_expanded = [t.replace("<image>", image_tokens_expanded) for t in texts]
      text_inputs = tokenizer(texts_expanded, return_tensors='pt',
                              padding=padding, truncation=truncation)
      image_inputs = image_processor(images=images, return_tensors='pt')
      return {
          "input_ids": text_inputs["input_ids"],
          "attention_mask": text_inputs["attention_mask"],
          "pixel_values": image_inputs["pixel_values"],
      }

  processor = _Processor(tokenizer, my_processor)

  # ---- 4) Data (validation split only; the test split is not used) ----
  print("\n[4/4] Loading data...")
  train_data = load_image_vqa_data('vqa_train_student.jsonl', answer_to_ids)
  val_data   = load_image_vqa_data('vqa_val_student_with_image_ids.jsonl', answer_to_ids)

  n_train_full = len(train_data['prompts'])
  n_subset = min(8000, n_train_full)
  indices = random.sample(range(n_train_full), n_subset)
  train_subset = {
      "image_paths": [train_data['image_paths'][i] for i in indices],
      "prompts":     [train_data['prompts'][i] for i in indices],
      "answers":     [train_data['answers'][i] for i in indices],
  }
  print(f"   Train: {n_subset} samples (out of {n_train_full})")
  print(f"   Val:   {len(val_data['prompts'])} samples")

  # ---- TRAINING (projector only) ----
  print("\n=== Training started ===")
  optimizer = torch.optim.AdamW(model.projector.parameters(), lr=1e-3)
  n_steps = (n_subset + batch_size - 1) // batch_size

  model.train()
  idx_perm = list(range(n_subset))
  random.shuffle(idx_perm)
  running_loss = 0.0
  log_every = 50

  for step in range(n_steps):
      bi = idx_perm[step * batch_size : (step + 1) * batch_size]
      paths   = [train_subset['image_paths'][i] for i in bi]
      prompts = [train_subset['prompts'][i]    for i in bi]
      anss    = [train_subset['answers'][i]    for i in bi]

      valid = [j for j in range(len(anss)) if anss[j] is not None]
      if not valid: continue
      paths   = [paths[j]   for j in valid]
      prompts = [prompts[j] for j in valid]
      anss    = [anss[j]    for j in valid]

      try:
          images = [Image.open(p).convert('RGB') for p in paths]
      except Exception as e:
          print(f"  Image read error step {step}: {e}"); continue

      full_texts = [prompts[j] + ['A','B','C','D'][anss[j]] for j in range(len(prompts))]
      inputs = my_processor(images=images, texts=full_texts)
      inputs = {k: v.to(device) for k, v in inputs.items()}
      if inputs["pixel_values"].dtype != dtype:
          inputs["pixel_values"] = inputs["pixel_values"].to(dtype)

      labels = inputs["input_ids"].clone()
      labels[labels == tokenizer.pad_token_id] = -100
      labels[labels == image_token_id] = -100

      outputs = model(input_ids=inputs["input_ids"],
                      attention_mask=inputs["attention_mask"],
                      pixel_values=inputs["pixel_values"],
                      labels=labels)
      loss = outputs.loss

      optimizer.zero_grad()
      loss.backward()
      torch.nn.utils.clip_grad_norm_(model.projector.parameters(), 1.0)
      optimizer.step()

      running_loss += loss.item()
      if (step + 1) % log_every == 0:
          print(f"  Step {step+1}/{n_steps} | Loss: {running_loss/log_every:.4f}")
          running_loss = 0.0

  print("=== Training finished ===\n")

  # ---- INFERENCE via get_prediction() ----
  print("Predicting on validation set...")
  final_val_predictions = get_prediction(model, processor, batch_size, device, val_data)

  write_answers_to_file('vqa_val_student_with_image_ids.jsonl', final_val_predictions, answer_to_ids)

  val_eval_acc = evaluation('vqa_val_teacher.jsonl', 'vqa_val_student_final.jsonl')
  print(f"\nFinal evaluation accuracy on val set: {val_eval_acc*100.0:.2f}%")

  # Expose these for reuse in Task 3 (and the accuracy itself, so the Task 3
  # summary table can reference the real measured number instead of a hardcoded one)
  return model, processor, tokenizer, image_processor, vision_tower.config, language_model.config, val_eval_acc


torch.cuda.empty_cache()
baseline_model, baseline_processor, baseline_tokenizer, _, _, _, baseline_val_acc = train_evaluate_LLaVA_model()

# Nothing downstream reuses the baseline model/processor/tokenizer (Task 3 rebuilds
# fresh models per experiment), so free the GPU memory now rather than holding onto
# it for the rest of the notebook - this is a shared 8GB GPU.
del baseline_model, baseline_processor, baseline_tokenizer
torch.cuda.empty_cache()

Device: cuda, dtype: torch.float16

[1/4] Loading vision tower (LongCLIP)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: AlpachinoNLP/LongCLIP-ViT-B-32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_

[2/4] Loading language model (Qwen3-0.6B)...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

   image_size=224, patch_size=32, num_patches=49
[3/4] Building custom LLaVA model...
   Vision hidden_states[-2] shape: torch.Size([1, 50, 768])  (expected [1, 50, 768])
   Trainable parameters: 1.84M / 685.07M total

[4/4] Loading data...
   Train: 8000 samples (out of 56055)
   Val:   3300 samples

=== Training started ===
  Step 50/500 | Loss: 2.4299
  Step 100/500 | Loss: 1.6488
  Step 150/500 | Loss: 1.4013
  Step 200/500 | Loss: 1.3359
  Step 250/500 | Loss: 1.2864
  Step 300/500 | Loss: 1.3094
  Step 350/500 | Loss: 1.2724
  Step 400/500 | Loss: 1.2370
  Step 450/500 | Loss: 1.2086
  Step 500/500 | Loss: 1.1982
=== Training finished ===

Predicting on validation set...
Written: vqa_val_student_final.jsonl (3300 samples)

Final evaluation accuracy on val set: 33.39%


## Task 3: Improving performance

Three strategies are compared against the Task 2 baseline, each through the same parameterised `run_experiment()` helper so that only one setting changes per run:

* **Architecture:** a deeper projector with LayerNorm and a residual connection (`projector_type`)
* **Fine-tuning:** LoRA on the Qwen3 attention layers (`use_lora`)
* **Data augmentation:** image augmentation on training samples (`use_augmentation`)


In [ ]:
#Your code for Task 3

import copy, time, shutil

# Install extra deps used below if missing
try:
    import peft
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'peft'], check=True)
    import peft

try:
    import torchvision
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'torchvision'], check=True)


class LlavaAdvancedProjector(nn.Module):
    """
    Architecture Tweak: a deeper, normalized projector than the basic
    2-layer MLP in `LlavaMultiModalProjector`. Differences:
      - LayerNorm on the raw CLIP features before projecting (CLIP hidden
        states are not scale-matched to the LM's embedding space).
      - 3 linear layers instead of 2, with an extra LayerNorm in the middle.
      - A residual connection around the last two layers, which helps
        gradient flow into the deeper layers when only the projector
        (a small fraction of total parameters) is being trained.
    Token count is unchanged (49 visual tokens in, 49 out) so it's a
    drop-in replacement for LlavaMultiModalProjector everywhere else.
    """

    def __init__(self, config: LlavaConfig):
        super().__init__()
        vision_dim = config.vision_config.hidden_size
        text_dim = config.text_config.hidden_size

        self.pre_norm = nn.LayerNorm(vision_dim)
        self.linear_in = nn.Linear(vision_dim, text_dim, bias=True)
        self.act1 = ACT2FN[config.projector_hidden_act]

        self.mid_norm = nn.LayerNorm(text_dim)
        self.linear_mid = nn.Linear(text_dim, text_dim, bias=True)
        self.act2 = ACT2FN[config.projector_hidden_act]
        self.linear_out = nn.Linear(text_dim, text_dim, bias=True)

    def forward(self, image_features):
        x = self.pre_norm(image_features)
        x = self.linear_in(x)
        x = self.act1(x)
        residual = x
        x = self.mid_norm(x)
        x = self.linear_mid(x)
        x = self.act2(x)
        x = self.linear_out(x)
        return x + residual


def run_experiment(
    projector_type="basic",      # "basic" or "advanced"
    use_lora=False,               # apply LoRA to the Qwen3 backbone
    lora_r=16,
    lora_alpha=32,
    lora_lr=2e-4,                 # LR for LoRA params (separate from projector LR)
    use_augmentation=False,       # image augmentation on training samples only
    lr=1e-3,                      # projector learning rate
    n_subset=8000,                # None -> use the full training set
    batch_size=16,                # EFFECTIVE batch size (controls LR schedule / step count)
    micro_batch_size=None,        # physical batch size actually sent to the GPU per forward/
                                   # backward call; None => same as batch_size (no accumulation).
                                   # Set this lower than batch_size on memory-constrained/shared
                                   # GPUs (e.g. micro_batch_size=2, batch_size=16) to keep the
                                   # same effective batch size / step count via gradient
                                   # accumulation while fitting in less VRAM per step.
    num_epochs=1,
    experiment_name="experiment",
):
    """
    Self-contained training + evaluation run for one Task 3 configuration.
    Builds a fresh CustomLlava model (frozen vision tower; projector always
    trainable; LM frozen unless use_lora=True), trains on the training set
    (optionally augmented), evaluates on the validation set only (the
    the test split is not used), and returns validation accuracy.
    """
    random.seed(0); torch.manual_seed(0); np.random.seed(0)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    dtype = torch.float16 if device == 'cuda' else torch.float32
    answer_to_ids = {'A': 0, 'B': 1, 'C': 2, 'D': 3}

    print(f"\n{'='*60}\nEXPERIMENT: {experiment_name}\n{'='*60}")
    print(f"  projector_type={projector_type}, use_lora={use_lora}, use_augmentation={use_augmentation}")
    if use_lora:
        print(f"  lora_r={lora_r}, lora_alpha={lora_alpha}, lora_lr={lora_lr}")

    # ---- Load components ----
    vision_tower = CLIPVisionModel.from_pretrained(CLIP_MODEL_ID, torch_dtype=dtype)
    image_processor_local = CLIPImageProcessor.from_pretrained(CLIP_MODEL_ID)
    language_model = AutoModelForCausalLM.from_pretrained(TEXI_MODEL_ID, torch_dtype=dtype)
    tokenizer = AutoTokenizer.from_pretrained(TEXI_MODEL_ID)

    if "<image>" not in tokenizer.get_vocab():
        tokenizer.add_special_tokens({"additional_special_tokens": ["<image>"]})
        language_model.resize_token_embeddings(len(tokenizer))
    image_token_id = tokenizer.convert_tokens_to_ids("<image>")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"  # predict()'s last_idx assumes this

    img_size = vision_tower.config.image_size
    patch_size = vision_tower.config.patch_size
    num_patches = (img_size // patch_size) ** 2

    # ---- Build projector (basic or advanced) ----
    class _Cfg: pass
    proj_cfg = _Cfg()
    proj_cfg.vision_config = vision_tower.config
    proj_cfg.text_config = language_model.config
    proj_cfg.projector_hidden_act = "gelu"

    if projector_type == "advanced":
        projector = LlavaAdvancedProjector(proj_cfg)
    else:
        projector = LlavaMultiModalProjector(proj_cfg)

    vision_tower = vision_tower.to(device).to(dtype)
    language_model = language_model.to(device).to(dtype)
    projector = projector.to(device).to(torch.float32)

    # ---- Apply LoRA if requested (Fine-tuning Technique) ----
    lora_params = []
    if use_lora:
        from peft import LoraConfig, get_peft_model, TaskType
        lora_cfg = LoraConfig(
            r=lora_r, lora_alpha=lora_alpha, lora_dropout=0.05,
            bias="none", task_type=TaskType.CAUSAL_LM,
            target_modules=["q_proj", "v_proj"],
        )
        language_model = get_peft_model(language_model, lora_cfg)
        lora_params = [p for p in language_model.parameters() if p.requires_grad]
        n_lora = sum(p.numel() for p in lora_params)
        print(f"  LoRA params: {n_lora/1e6:.2f}M")

    model = CustomLlava(vision_tower, language_model, projector, image_token_id, vision_feature_layer=-2)

    for p in model.vision_tower.parameters():
        p.requires_grad = False
    for p in model.projector.parameters():
        p.requires_grad = True
    if not use_lora:
        for p in model.language_model.parameters():
            p.requires_grad = False

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable: {trainable/1e6:.2f}M params")

    # ---- Data (val only) ----
    train_data = load_image_vqa_data('vqa_train_student.jsonl', answer_to_ids)
    val_data   = load_image_vqa_data('vqa_val_student_with_image_ids.jsonl', answer_to_ids)

    n_train_full = len(train_data['prompts'])
    n_use = n_train_full if n_subset is None else min(n_subset, n_train_full)
    indices = random.sample(range(n_train_full), n_use)
    train_subset = {k: [train_data[k][i] for i in indices] for k in train_data}
    print(f"  Train: {n_use} samples (of {n_train_full}) | Val: {len(val_data['prompts'])} samples")

    # ---- Data Augmentation (train images only, never applied at eval time) ----
    train_augment = None
    if use_augmentation:
        import torchvision.transforms as T
        train_augment = T.Compose([
            T.RandomHorizontalFlip(p=0.5),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            T.RandomResizedCrop(size=(img_size, img_size), scale=(0.85, 1.0)),
        ])
        print("  Image augmentation: RandomHorizontalFlip + ColorJitter + RandomResizedCrop")
        print("  NOTE: horizontal flip can hurt questions where left/right or reading")
        print("  order matters - worth checking per-category accuracy if this backfires.")

    image_tokens_expanded = "<image>" * num_patches

    def my_processor(images, texts, augment=False):
        if augment and train_augment is not None:
            images = [train_augment(img) for img in images]
        texts_exp = [t.replace("<image>", image_tokens_expanded) for t in texts]
        ti = tokenizer(texts_exp, return_tensors='pt', padding=True, truncation=True)
        ii = image_processor_local(images=images, return_tensors='pt')
        return {"input_ids": ti["input_ids"], "attention_mask": ti["attention_mask"],
                "pixel_values": ii["pixel_values"]}

    # ---- Optimizer ----
    param_groups = [{"params": model.projector.parameters(), "lr": lr}]
    if use_lora and lora_params:
        param_groups.append({"params": lora_params, "lr": lora_lr})
    optimizer = torch.optim.AdamW(param_groups)

    micro_bs = micro_batch_size or batch_size
    accum_steps = max(1, batch_size // micro_bs)
    n_steps_per_epoch = (n_use + batch_size - 1) // batch_size
    n_steps = n_steps_per_epoch * num_epochs
    if micro_bs != batch_size:
        print(f"  Gradient accumulation: micro_batch_size={micro_bs} x {accum_steps} "
              f"-> effective batch_size={micro_bs * accum_steps}")

    def _forward_loss(bi):
        """Runs one micro-batch forward pass and returns the (unscaled) loss,
        or None if every sample in this micro-batch had no gold answer."""
        paths = [train_subset['image_paths'][i] for i in bi]
        prompts = [train_subset['prompts'][i] for i in bi]
        anss = [train_subset['answers'][i] for i in bi]
        valid = [j for j in range(len(anss)) if anss[j] is not None]
        if not valid:
            return None
        paths = [paths[j] for j in valid]
        prompts = [prompts[j] for j in valid]
        anss = [anss[j] for j in valid]

        try:
            imgs = [Image.open(p).convert('RGB') for p in paths]
        except Exception:
            return None

        full = [prompts[j] + ['A', 'B', 'C', 'D'][anss[j]] for j in range(len(prompts))]
        inputs = my_processor(images=imgs, texts=full, augment=use_augmentation)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        if inputs["pixel_values"].dtype != dtype:
            inputs["pixel_values"] = inputs["pixel_values"].to(dtype)

        labels = inputs["input_ids"].clone()
        labels[labels == tokenizer.pad_token_id] = -100
        labels[labels == image_token_id] = -100

        outputs = model(input_ids=inputs["input_ids"],
                        attention_mask=inputs["attention_mask"],
                        pixel_values=inputs["pixel_values"], labels=labels)
        return outputs.loss

    # ---- Training ----
    print(f"  Training for {num_epochs} epoch(s), {n_steps} total steps...")
    t0 = time.time()
    model.train()
    running_loss = 0.0; log_every = 100; global_step = 0

    for epoch in range(num_epochs):
        idx_perm = list(range(n_use)); random.shuffle(idx_perm)
        for step in range(n_steps_per_epoch):
            bi_full = idx_perm[step * batch_size:(step + 1) * batch_size]
            if not bi_full:
                continue

            optimizer.zero_grad()
            step_loss = 0.0
            n_micro_done = 0
            for mb_start in range(0, len(bi_full), micro_bs):
                bi = bi_full[mb_start:mb_start + micro_bs]
                loss = _forward_loss(bi)
                if loss is None:
                    continue
                (loss / accum_steps).backward()
                step_loss += loss.item()
                n_micro_done += 1

            if n_micro_done == 0:
                continue

            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step()

            running_loss += step_loss / n_micro_done; global_step += 1
            if global_step % log_every == 0:
                print(f"    Step {global_step}/{n_steps} | Loss: {running_loss/log_every:.4f}")
                running_loss = 0.0

    print(f"  Training time: {(time.time()-t0)/60:.1f} min")

    # ---- Evaluation (val only, never augmented) ----
    def predict(dataset):
        model.eval()
        ips, prs = dataset["image_paths"], dataset["prompts"]
        n = len(prs)
        abcd = []
        for l in ["A", "B", "C", "D"]:
            ids = tokenizer(l, add_special_tokens=False).input_ids
            assert len(ids) == 1, f"{l!r} is not a single token: {ids}"
            abcd.append(ids[0])
        preds = []
        with torch.no_grad():
            for s in range(0, n, micro_bs):
                e = min(s + micro_bs, n)
                imgs = [Image.open(p).convert('RGB') for p in ips[s:e]]
                inputs = my_processor(images=imgs, texts=prs[s:e], augment=False)
                inputs = {k: v.to(device) for k, v in inputs.items()}
                if inputs["pixel_values"].dtype != dtype:
                    inputs["pixel_values"] = inputs["pixel_values"].to(dtype)
                outputs = model(input_ids=inputs["input_ids"],
                                attention_mask=inputs["attention_mask"],
                                pixel_values=inputs["pixel_values"])
                logits = outputs.logits
                last_idx = inputs["attention_mask"].sum(dim=1) - 1
                for i in range(logits.size(0)):
                    last = logits[i, last_idx[i], :]
                    scores = torch.tensor([last[t].item() for t in abcd])
                    preds.append(int(scores.argmax().item()))
        return preds

    val_preds = predict(val_data)
    write_answers_to_file('vqa_val_student_with_image_ids.jsonl', val_preds, answer_to_ids)
    val_acc = evaluation('vqa_val_teacher.jsonl', 'vqa_val_student_final.jsonl')
    print(f"  >>> {experiment_name} val accuracy: {val_acc*100:.2f}% <<<")

    # keep a tagged copy of this experiment's predictions for later comparison
    shutil.copy('vqa_val_student_final.jsonl', f'vqa_val_{experiment_name}.jsonl')

    del model, vision_tower, language_model, projector
    torch.cuda.empty_cache()
    return val_acc

#### 3.1 Architecture Tweak: advanced projector

Swap the basic 2-layer MLP for `LlavaAdvancedProjector` (LayerNorm + 3 linear layers + residual), everything else identical to the Task 2 baseline (no LoRA, no augmentation, same `n_subset`/`batch_size`/`lr`) so the accuracy delta is attributable to the projector change alone.

In [ ]:
acc_advanced_projector = run_experiment(
    projector_type="advanced",
    n_subset=8000,
    batch_size=16,       # same effective batch size as the Task 2 baseline, for a fair comparison
    num_epochs=1,
    lr=1e-3,
    experiment_name="advanced_projector",
)


EXPERIMENT: advanced_projector
  projector_type=advanced, use_lora=False, use_augmentation=False


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: AlpachinoNLP/LongCLIP-ViT-B-32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

  Trainable: 2.89M params
  Train: 8000 samples (of 56055) | Val: 3300 samples
  Training for 1 epoch(s), 500 total steps...
    Step 100/500 | Loss: 1.3085
    Step 200/500 | Loss: 1.1006
    Step 300/500 | Loss: 1.0963
    Step 400/500 | Loss: 1.0844
    Step 500/500 | Loss: 1.0575
  Training time: 1.6 min
Written: vqa_val_student_final.jsonl (3300 samples)
  >>> advanced_projector val accuracy: 40.82% <<<


**Result:** the advanced projector achieved **40.82%** validation accuracy, compared with **33.39%** for the basic-projector baseline - an absolute improvement of **+7.43 points**. Both experiments used the same 8000-sample subset, effective batch size, learning rate, and 500-step training budget, so the improvement can reasonably be attributed to the deeper, normalised projector architecture (LayerNorm + third linear layer + residual, 2.89M vs 1.84M trainable parameters) rather than to a difference in training budget.

#### 3.2 Fine-tuning Technique: LoRA on the Qwen3 backbone

`target_modules=["q_proj", "v_proj"]`, `r=16`, `alpha=32` with a lower LR (2e-4) than the projector (1e-3), trained on the **full** training set for 2 epochs. These hyperparameters were carried over from the strongest configuration found during development on the earlier assignment 2 notebook.

**Result:** LoRA fine-tuning achieved **70.00%** validation accuracy, the strongest result among the three Task 3 strategies - an absolute improvement of **+36.61 points** over the 33.39% Task 2 baseline.

**Fairness caveat:** unlike the advanced-projector and augmentation experiments above/below (which keep the same 8000-sample subset, 500-step budget, and projector-only training as the baseline), this LoRA run trains on the **full 56,055-sample training set for 2 epochs (7,008 steps)** and updates LoRA parameters in the language model backbone in addition to the projector. So the +36.61 point gain reflects the *combined* effect of parameter-efficient LM adaptation, ~7x more training data, and a much larger step budget - it should not be attributed to LoRA alone. The strongest controlled comparison would rerun LoRA at the same 8000-sample/500-step budget as the other two strategies; given that run takes roughly three hours on this shared GPU, this notebook reports the stronger, larger-budget configuration instead and flags the caveat here rather than re-running it.

**GPU note:** this shared jhub GPU (8GB) OOMs at `batch_size=16` run directly. `micro_batch_size=2` below makes the physical forward/backward batch 2 (fits in memory) while `batch_size=16` is kept as the *effective* batch size via gradient accumulation (8 micro-steps per optimizer step) - so the step count (~7008) and training dynamics match a true batch-16 run, only the physical memory footprint changes. If this is still too slow given jhub's session time limit, interrupt and re-run with a smaller `n_subset` (e.g. 16000-24000) instead of `None`.
> **Note:** the saved output of the training cell for this run is incomplete in this notebook, so the figure above is not reproduced in the cell output below.


In [ ]:
!pip install -q --upgrade torchao peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 55.3 MB/s eta 0:00:00


In [ ]:
acc_lora = run_experiment(
    projector_type="basic",
    use_lora=True,
    lora_r=16,
    lora_alpha=32,
    lora_lr=2e-4,
    n_subset=None,       # full training set (56055 samples), as in the original run
    batch_size=16,       # effective batch size - keeps the original ~7008-step schedule
    micro_batch_size=2,  # physical batch size actually sent to the GPU (fits in 8GB)
    num_epochs=2,
    experiment_name="lora_finetuning",
)


EXPERIMENT: lora_finetuning
  projector_type=basic, use_lora=True, use_augmentation=False
  lora_r=16, lora_alpha=32, lora_lr=0.0002


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: AlpachinoNLP/LongCLIP-ViT-B-32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

  LoRA params: 2.29M
  Trainable: 4.13M params
  Train: 56055 samples (of 56055) | Val: 3300 samples
  Gradient accumulation: micro_batch_size=2 x 8 -> effective batch_size=16
  Training for 2 epoch(s), 7008 total steps...
    Step 100/7008 | Loss: 1.2663
    Step 200/7008 | Loss: 1.0105
    Step 300/7008 | Loss: 0.9532
    Step 400/7008 | Loss: 0.9137
    Step 500/7008 | Loss: 0.9080
    Step 600/7008 | Loss: 0.8801
    Step 700/7008 | Loss: 0.8881
    Step 800/7008 | Loss: 0.8703
    Step 900/7008 | Loss: 0.8787
    Step 1000/7008 | Loss: 0.8588
    Step 1100/7008 | Loss: 0.8476
    Step 1200/7008 | Loss: 0.8630
    Step 1300/7008 | Loss: 0.8414
    Step 1400/7008 | Loss: 0.8358
    Step 1500/7008 | Loss: 0.8383
    Step 1600/7008 | Loss: 0.8303
    Step 1700/7008 | Loss: 0.8316
    Step 1800/7008 | Loss: 0.8466
    Step 1900/7008 | Loss: 0.8245
    Step 2000/7008 | Loss: 0.8142
    Step 2100/7008 | Loss: 0.8447
    Step 2200/7008 | Loss: 0.8620
    Step 2300/7008 | Loss: 0.8171
    

KeyboardInterrupt: 

#### 3.3 Data Augmentation: image augmentation on training samples

`RandomHorizontalFlip` + `ColorJitter` + `RandomResizedCrop`, applied only to training images (never at evaluation time), otherwise identical setup to the Task 2 baseline.

In [ ]:
acc_augmentation = run_experiment(
    projector_type="basic",
    use_augmentation=True,
    n_subset=8000,
    batch_size=16,      # same effective batch size as the baseline/advanced-projector runs,
                        # so the only changed variable is augmentation, not step count
    micro_batch_size=2, # physical batch size that fits this GPU; batch_size=16 is kept as the
                        # effective size via gradient accumulation
    num_epochs=1,
    lr=1e-3,
    experiment_name="data_augmentation",
)

**Result:** image augmentation achieved **42.55%** validation accuracy, an absolute improvement of **+9.16 points** over the 33.39% baseline, using the same 8000-sample subset and 500-step budget as the baseline and advanced-projector experiments (fair comparison, only the augmentation knob changed). This suggests moderate appearance and crop variation improved visual robustness. However, `RandomHorizontalFlip` in particular could be harmful for questions involving left/right relationships, on-image text, or directional reasoning - a per-category accuracy breakdown would be needed to confirm whether this is happening, which is left as a possible follow-up.
> **Note:** the saved output of the training cell for this run is incomplete in this notebook, so the figure above is not reproduced in the cell output below.


#### Summary

Validation accuracy by category, all measured against the same `vqa_val_teacher.jsonl` gold file:

In [ ]:
print(f"{'Baseline (Task 2, basic projector)':<45}: {baseline_val_acc*100:.2f}%")
print(f"{'Architecture Tweak (advanced projector)':<45}: {acc_advanced_projector*100:.2f}%")
print(f"{'Fine-tuning (LoRA)':<45}: {acc_lora*100:.2f}%")
print(f"{'Data Augmentation':<45}: {acc_augmentation*100:.2f}%")